# ============================================================
# 02_OULAD_Feature_Engineering.ipynb
# DropGuard — Feature Engineering Enrichi
# Features Cumulatives + Récentes (14j) + Tendance
# IRD / UGB Saint-Louis — 2026
# ============================================================

In [4]:
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import pandas as pd
import numpy as np
import os
import gc
import json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports OK")

✅ Imports OK


In [8]:
BASE_PATH      = '/content/drive/MyDrive/MOOC_Recommandation_Adaptative'
RAW_PATH       = f"{BASE_PATH}/data/raw/OULAD"
PROCESSED_PATH = f"{RAW_PATH}/processed"
RESULTS_PATH   = f"{BASE_PATH}/results"

os.makedirs(PROCESSED_PATH, exist_ok=True)

# Paramètres globaux
CUTOFFS       = [28, 56, 84, 112, 140]
WINDOW_RECENT = 14   # fenêtre glissante récente
MIN_DAYS      = 7    # minimum jours avant prédiction

print(f"Cutoffs      : {CUTOFFS}")
print(f"Fenêtre récente : {WINDOW_RECENT} jours")
print(f"Minimum jours  : {MIN_DAYS} jours")
print("✅ Chemins configurés")

Cutoffs      : [28, 56, 84, 112, 140]
Fenêtre récente : 14 jours
Minimum jours  : 7 jours
✅ Chemins configurés


In [9]:
df_student_info        = pd.read_csv(f"{RAW_PATH}/studentInfo.csv")
df_student_vle         = pd.read_csv(f"{RAW_PATH}/studentVle.csv")
df_student_assessments = pd.read_csv(f"{RAW_PATH}/studentAssessment.csv")
df_assessments         = pd.read_csv(f"{RAW_PATH}/assessments.csv")
df_vle                 = pd.read_csv(f"{RAW_PATH}/vle.csv")

# Variable cible
df_student_info['is_withdrawn'] = (
    df_student_info['final_result'] == 'Withdrawn'
).astype(int)

# Enrichir VLE avec activity_type
df_student_vle = pd.merge(
    df_student_vle,
    df_vle[['id_site', 'activity_type']],
    on='id_site', how='left'
)

# Enrichir assessments
df_student_assessments = pd.merge(
    df_student_assessments,
    df_assessments[['id_assessment','assessment_type','date','weight']],
    on='id_assessment'
)
df_student_assessments.rename(
    columns={'date': 'date_due'}, inplace=True
)

print(f"Étudiants    : {len(df_student_info):,}")
print(f"Interactions : {len(df_student_vle):,}")
print(f"Assessments  : {len(df_student_assessments):,}")
print(f"Taux abandon : {df_student_info['is_withdrawn'].mean()*100:.1f}%")
print(f"Jours max    : {df_student_vle['date'].max()}")
print("✅ Données chargées")

Étudiants    : 32,593
Interactions : 10,655,280
Assessments  : 173,912
Taux abandon : 31.2%
Jours max    : 269
✅ Données chargées


In [10]:
def build_features_enriched(cutoff_day, window_recent=14):
    """
    Construit les features enrichies pour un cutoff donné.

    Features cumulatives : J1 → cutoff_day
    Features récentes    : (cutoff_day - window_recent) → cutoff_day
    Features dérivées    : tendance, jours_inactivite, cutoff_day

    Args:
        cutoff_day    : jour limite (ex: 28, 56, 84...)
        window_recent : fenêtre récente en jours (défaut: 14)

    Returns:
        DataFrame enrichi avec toutes les features
    """
    print(f"\n{'='*60}")
    print(f"  CUTOFF : Jour {cutoff_day} | Fenêtre récente : {window_recent}j")
    print(f"{'='*60}")

    recent_start = cutoff_day - window_recent

    # ──────────────────────────────────────────
    # PARTIE 1 — FEATURES CUMULATIVES (J1 → cutoff_day)
    # ──────────────────────────────────────────
    vle_cumul = df_student_vle[
        df_student_vle['date'] <= cutoff_day
    ].copy()

    print(f"  VLE cumulatif   : {len(vle_cumul):,} interactions")

    # Engagement cumulatif
    engagement_cumul = vle_cumul.groupby(
        ['id_student','code_module','code_presentation']
    ).agg(
        total_clicks       = ('sum_click','sum'),
        total_days_active  = ('date','nunique'),
        last_activity_day  = ('date','max'),
        first_activity_day = ('date','min'),
    ).reset_index()

    engagement_cumul['avg_clicks_per_day'] = (
        engagement_cumul['total_clicks'] /
        engagement_cumul['total_days_active'].replace(0,1)
    )
    engagement_cumul['total_span_days'] = (
        engagement_cumul['last_activity_day'] -
        engagement_cumul['first_activity_day']
    )

    # Activités VLE cumulatives par type
    activity_cumul = pd.pivot_table(
        vle_cumul,
        values='sum_click',
        index=['id_student','code_module','code_presentation'],
        columns='activity_type',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

    engagement_cumul = pd.merge(
        engagement_cumul, activity_cumul,
        on=['id_student','code_module','code_presentation'],
        how='left'
    )

    # ──────────────────────────────────────────
    # PARTIE 2 — FEATURES RÉCENTES (cutoff-14j → cutoff_day)
    # ──────────────────────────────────────────
    vle_recent = df_student_vle[
        (df_student_vle['date'] > recent_start) &
        (df_student_vle['date'] <= cutoff_day)
    ].copy()

    print(f"  VLE récent      : {len(vle_recent):,} interactions")

    engagement_recent = vle_recent.groupby(
        ['id_student','code_module','code_presentation']
    ).agg(
        clicks_recent      = ('sum_click','sum'),
        days_active_recent = ('date','nunique'),
    ).reset_index()

    # Activités VLE récentes par type
    if len(vle_recent) > 0:
        activity_recent = pd.pivot_table(
            vle_recent,
            values='sum_click',
            index=['id_student','code_module','code_presentation'],
            columns='activity_type',
            aggfunc='sum',
            fill_value=0
        ).reset_index()

        act_cols = [c for c in activity_recent.columns
                    if c not in ['id_student','code_module',
                                 'code_presentation']]
        activity_recent = activity_recent.rename(
            columns={c: f"{c}_recent" for c in act_cols}
        )

        engagement_recent = pd.merge(
            engagement_recent, activity_recent,
            on=['id_student','code_module','code_presentation'],
            how='left'
        )

    # ──────────────────────────────────────────
    # PARTIE 3 — ASSESSMENTS CUMULATIFS
    # ──────────────────────────────────────────
    assess = df_student_assessments[
        df_student_assessments['date_submitted'] <= cutoff_day
    ].copy()

    print(f"  Assessments     : {len(assess):,} soumissions")

    if len(assess) > 0:
        performance = assess.groupby('id_student').agg(
            avg_score                 = ('score','mean'),
            num_submitted_assessments = ('id_assessment','count')
        ).reset_index()

        avg_by_type = pd.pivot_table(
            assess,
            values='score',
            index='id_student',
            columns='assessment_type',
            aggfunc='mean',
            fill_value=0
        ).reset_index()

        performance = pd.merge(
            performance, avg_by_type,
            on='id_student', how='left'
        )
    else:
        performance = df_student_info[['id_student']].copy()
        performance['avg_score'] = 0
        performance['num_submitted_assessments'] = 0

    # ──────────────────────────────────────────
    # PARTIE 4 — FUSION FINALE
    # ──────────────────────────────────────────
    final_df = pd.merge(
        df_student_info,
        engagement_cumul,
        on=['id_student','code_module','code_presentation'],
        how='left'
    )
    final_df = pd.merge(
        final_df, engagement_recent,
        on=['id_student','code_module','code_presentation'],
        how='left'
    )
    final_df = pd.merge(
        final_df, performance,
        on='id_student', how='left'
    )

    # Remplir NaN numériques par 0
    num_cols = final_df.select_dtypes(
        include=['float64','int64']
    ).columns.tolist()
    for col in num_cols:
        if col not in ['id_student','is_withdrawn']:
            final_df[col] = final_df[col].fillna(0)

    # ──────────────────────────────────────────
    # PARTIE 5 — FEATURES DÉRIVÉES
    # ──────────────────────────────────────────
    # Tendance : ratio activité récente / cumulative
    final_df['tendance_activite'] = (
        final_df.get('clicks_recent', 0) /
        final_df['total_clicks'].replace(0, 1)
    ).clip(0, 1)

    # Jours depuis dernière activité
    final_df['jours_inactivite'] = (
        cutoff_day - final_df['last_activity_day']
    ).clip(0, cutoff_day)

    # Cutoff comme feature
    final_df['cutoff_day'] = cutoff_day

    print(f"  Dataset final   : {final_df.shape}")
    print(f"  Taux abandon    : {final_df['is_withdrawn'].mean()*100:.1f}%")
    print(f"  Tendance moy.   : {final_df['tendance_activite'].mean():.3f}")
    print(f"  Inactivité moy. : {final_df['jours_inactivite'].mean():.1f}j")

    return final_df

print("✅ Fonction build_features_enriched() définie")

✅ Fonction build_features_enriched() définie


In [11]:
print("="*60)
print("  GÉNÉRATION DES 5 DATASETS")
print("="*60)

datasets = []

for cutoff in CUTOFFS:
    df_c = build_features_enriched(cutoff, WINDOW_RECENT)
    datasets.append(df_c)

    path = f"{PROCESSED_PATH}/oulad_enriched_cutoff_{cutoff}.csv"
    df_c.to_csv(path, index=False)
    print(f"  Sauvegardé → cutoff_{cutoff}.csv")
    gc.collect()

print(f"\n✅ {len(CUTOFFS)} datasets générés")

  GÉNÉRATION DES 5 DATASETS

  CUTOFF : Jour 28 | Fenêtre récente : 14j
  VLE cumulatif   : 2,847,851 interactions
  VLE récent      : 1,028,074 interactions
  Assessments     : 25,614 soumissions
  Dataset final   : (32593, 64)
  Taux abandon    : 31.2%
  Tendance moy.   : 0.337
  Inactivité moy. : 7.6j
  Sauvegardé → cutoff_28.csv

  CUTOFF : Jour 56 | Fenêtre récente : 14j
  VLE cumulatif   : 4,331,544 interactions
  VLE récent      : 728,498 interactions
  Assessments     : 51,578 soumissions
  Dataset final   : (32593, 66)
  Taux abandon    : 31.2%
  Tendance moy.   : 0.135
  Inactivité moy. : 14.7j
  Sauvegardé → cutoff_56.csv

  CUTOFF : Jour 84 | Fenêtre récente : 14j
  VLE cumulatif   : 5,268,730 interactions
  VLE récent      : 388,548 interactions
  Assessments     : 66,984 soumissions
  Dataset final   : (32593, 65)
  Taux abandon    : 31.2%
  Tendance moy.   : 0.048
  Inactivité moy. : 25.3j
  Sauvegardé → cutoff_84.csv

  CUTOFF : Jour 112 | Fenêtre récente : 14j
  VLE cu

In [12]:
print("="*60)
print("  HARMONISATION ET DATASET COMBINÉ")
print("="*60)

# Colonnes communes à tous les datasets
all_col_sets = [set(df.columns) for df in datasets]
common_cols  = set.intersection(*all_col_sets)

# Colonnes à exclure des features
EXCLUDE = {
    'id_student','code_module','code_presentation',
    'final_result','is_withdrawn','cutoff_day',
    'gender','region','highest_education',
    'imd_band','age_band','disability',
    'num_of_prev_attempts','studied_credits',
    'date_registration','date_unregistration',
    'code_module_x','code_presentation_x'
}

# Features retenues
FEATURE_COLS = sorted([
    c for c in common_cols
    if c not in EXCLUDE
    and datasets[0][c].dtype in ['float64','int64','int32','float32']
])

print(f"\nFeatures communes        : {len(FEATURE_COLS)}")
print(f"  Cumulatives            : {len([c for c in FEATURE_COLS if '_recent' not in c and c not in ['cutoff_day','tendance_activite','jours_inactivite']])}")
print(f"  Récentes (14j)         : {len([c for c in FEATURE_COLS if '_recent' in c])}")
print(f"  Dérivées               : tendance_activite, jours_inactivite")

# Dataset combiné — garder seulement colonnes communes
datasets_harmonized = [df[list(common_cols)] for df in datasets]
df_combined = pd.concat(datasets_harmonized, ignore_index=True)

# Sauvegarde
df_combined.to_csv(
    f"{PROCESSED_PATH}/oulad_combined_final.csv", index=False
)

# Sauvegarder liste features
with open(f"{PROCESSED_PATH}/feature_cols_final.json", 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)

print(f"\nDataset combiné final :")
print(f"  Lignes   : {len(df_combined):,}")
print(f"  Colonnes : {df_combined.shape[1]}")
print(f"  Taux abandon : {df_combined['is_withdrawn'].mean()*100:.1f}%")
print(f"  Cutoffs  : {sorted(df_combined['cutoff_day'].unique())}")
print(f"\n✅ Dataset combiné sauvegardé")

  HARMONISATION ET DATASET COMBINÉ

Features communes        : 50
  Cumulatives            : 28
  Récentes (14j)         : 20
  Dérivées               : tendance_activite, jours_inactivite

Dataset combiné final :
  Lignes   : 162,965
  Colonnes : 64
  Taux abandon : 31.2%
  Cutoffs  : [np.int64(28), np.int64(56), np.int64(84), np.int64(112), np.int64(140)]

✅ Dataset combiné sauvegardé


In [13]:
print("="*60)
print("  RAPPORT COMPARATIF DES CUTOFFS")
print("="*60)
print(f"\n{'Cutoff':<10} {'Étudiants':<12} {'Abandon':<10} {'Tendance':<12} {'Inactivité'}")
print("-"*56)

for cutoff, df in zip(CUTOFFS, datasets):
    n      = len(df)
    taux   = df['is_withdrawn'].mean()*100
    tend   = df['tendance_activite'].mean()
    inact  = df['jours_inactivite'].mean()
    print(f"Jour {cutoff:<6} {n:<12,} {taux:<10.1f}% {tend:<12.3f} {inact:.1f}j")

print(f"\n{'='*56}")
print(f"Dataset combiné : {len(df_combined):,} lignes")
print(f"Features finales : {len(FEATURE_COLS)}")
print(f"\n✅ NB02 terminé — Prêt pour NB03")

  RAPPORT COMPARATIF DES CUTOFFS

Cutoff     Étudiants    Abandon    Tendance     Inactivité
--------------------------------------------------------
Jour 28     32,593       31.2      % 0.337        7.6j
Jour 56     32,593       31.2      % 0.135        14.7j
Jour 84     32,593       31.2      % 0.048        25.3j
Jour 112    32,593       31.2      % 0.057        32.5j
Jour 140    32,593       31.2      % 0.061        42.5j

Dataset combiné : 162,965 lignes
Features finales : 50

✅ NB02 terminé — Prêt pour NB03
